Folium chart

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  
import math

df = pd.read_csv('/Volumes/Extreme Pro/DSAN-5200/final-project/T_DB1B_MARKET_CLEAN.csv')   


note that this folium map only looks at the top 500 busiest routes!!!

In [13]:
import folium
import pandas as pd
import branca.colormap as cm


# Aggregate route-level data
route_df = (
    df.groupby(
        ["ORIGIN", "DEST", "ORIGIN_LAT", "ORIGIN_LON", "DEST_LAT", "DEST_LON"], ## group by unique flight routes
        as_index=False
    )
    .agg(
        avg_fare=("MARKET_FARE", "mean"),       # avg ticket price
        avg_distance=("MARKET_DISTANCE", "mean"),   # avg flight distance
        passengers=("PASSENGERS", "sum")        # total number of passengers -- is this useful....
    )
)

# Keep only TOP 500 BUSIEST ROUTES
route_df = (
    route_df
    .dropna(subset=["ORIGIN_LAT", "ORIGIN_LON", "DEST_LAT", "DEST_LON", "avg_fare"])
    .sort_values("passengers", ascending=False)
    .head(500) 
)

# Base map
m = folium.Map(
    location=[39.5, -98.35],
    zoom_start=4,
    tiles="CartoDB positron",
    control_scale=True
)

# Color scale
colormap = cm.LinearColormap(
    colors=["#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"], # COLORS CAN CHANGE HERE
    vmin=route_df["avg_fare"].quantile(0.05),
    vmax=route_df["avg_fare"].quantile(0.95),
    caption="Average Fare ($)"
)

p90 = route_df["passengers"].quantile(0.90)

# Routes
for _, row in route_df.iterrows():

    fare_color = colormap(row["avg_fare"])

    line_weight = max(1.2, min(7, row["passengers"] / p90 * 5))

    #  Tooltip 
    tooltip_text = (
        f"{row['ORIGIN']} → {row['DEST']} | "
        f"${row['avg_fare']:,.0f} | "
        f"{row['avg_distance']:,.0f} miles | "
        f"{row['passengers']:,.0f} passengers" ## again...could get rid of this
    )

    #  Popup 
    popup_html = f"""
    <div style="font-family: sans-serif; width: 220px;">
        <h4 style="margin-bottom:6px;">
            {row['ORIGIN']} → {row['DEST']}
        </h4>
        <b>Average fare:</b> ${row['avg_fare']:,.0f}<br>
        <b>Distance:</b> {row['avg_distance']:,.0f} miles<br>
        <b>Passengers:</b> {row['passengers']:,.0f}
    </div>
    """

    # draw the line
    folium.PolyLine(
        locations=[
            [row["ORIGIN_LAT"], row["ORIGIN_LON"]],
            [row["DEST_LAT"], row["DEST_LON"]]
        ],
        color=fare_color,
        weight=line_weight,
        opacity=0.55,
        tooltip=tooltip_text,
        popup=folium.Popup(popup_html, max_width=260)
    ).add_to(m)

# Airport markers
airport_df = pd.concat([
    route_df[["ORIGIN", "ORIGIN_LAT", "ORIGIN_LON"]]
        .rename(columns={"ORIGIN": "airport", "ORIGIN_LAT": "lat", "ORIGIN_LON": "lon"}),
    route_df[["DEST", "DEST_LAT", "DEST_LON"]]
        .rename(columns={"DEST": "airport", "DEST_LAT": "lat", "DEST_LON": "lon"})
]).drop_duplicates()

for _, row in airport_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3.5,
        color="#264653",
        fill=True,
        fill_color="#264653",
        fill_opacity=0.85,
        tooltip=row["airport"]
    ).add_to(m)

# Legend
colormap.add_to(m)

m

In [12]:
## with dropdwon option

import ipywidgets as widgets
from IPython.display import display, clear_output

airport_options = sorted(
    set(route_df["ORIGIN"]).union(set(route_df["DEST"]))
) 

# dropdown menu
airport_dropdown = widgets.Dropdown(
    options=["All Airports"] + airport_options,
    value="All Airports",
    description="Airport:"
)

def make_route_map(selected_airport):
    if selected_airport == "All Airports":
        filtered_routes = route_df.copy()
    else:
        filtered_routes = route_df[
            (route_df["ORIGIN"] == selected_airport) |
            (route_df["DEST"] == selected_airport)
        ].copy()

## map is essentially the same as above
    m = folium.Map(
        location=[39.5, -98.35],
        zoom_start=4,
        tiles="CartoDB positron",
        control_scale=True
    )

    colormap = cm.LinearColormap(
        colors=["#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"],
        vmin=route_df["avg_fare"].quantile(0.05),
        vmax=route_df["avg_fare"].quantile(0.95),
        caption="Average Fare ($)"
    )

    p90 = route_df["passengers"].quantile(0.90)

    for _, row in filtered_routes.iterrows():
        tooltip_text = (
            f"{row['ORIGIN']} → {row['DEST']} | "
            f"${row['avg_fare']:,.0f} | "
            f"{row['avg_distance']:,.0f} miles | "
            f"{row['passengers']:,.0f} passengers"
        )

        popup_html = f"""
        <div style="font-family: sans-serif; width: 220px;">
            <h4>{row['ORIGIN']} → {row['DEST']}</h4>
            <b>Fare:</b> ${row['avg_fare']:,.0f}<br>
            <b>Distance:</b> {row['avg_distance']:,.0f} mi<br>
            <b>Passengers:</b> {row['passengers']:,.0f}
        </div>
        """

        folium.PolyLine(
            locations=[
                [row["ORIGIN_LAT"], row["ORIGIN_LON"]],
                [row["DEST_LAT"], row["DEST_LON"]]
            ],
            color=colormap(row["avg_fare"]),
            weight=max(1.2, min(7, row["passengers"] / p90 * 5)),
            opacity=0.6,
            tooltip=tooltip_text,
            popup=folium.Popup(popup_html, max_width=260)
        ).add_to(m)

    airport_df = pd.concat([
        filtered_routes[["ORIGIN", "ORIGIN_LAT", "ORIGIN_LON"]]
            .rename(columns={"ORIGIN": "airport", "ORIGIN_LAT": "lat", "ORIGIN_LON": "lon"}),
        filtered_routes[["DEST", "DEST_LAT", "DEST_LON"]]
            .rename(columns={"DEST": "airport", "DEST_LAT": "lat", "DEST_LON": "lon"})
    ]).drop_duplicates()

    for _, row in airport_df.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=4,
            color="#1d3557",
            fill=True,
            fill_color="#1d3557",
            fill_opacity=0.9,
            weight=1,
            tooltip=row["airport"]
        ).add_to(m)

    colormap.add_to(m)
    return m


output = widgets.Output()

def update_map(change=None):
    with output:
        clear_output(wait=True)
        display(make_route_map(airport_dropdown.value))

airport_dropdown.observe(update_map, names="value")

display(airport_dropdown, output)
update_map()

Dropdown(description='Airport:', options=('All Airports', 'ATL', 'AUS', 'BDL', 'BNA', 'BOS', 'BUF', 'BUR', 'BW…

Output()

In [16]:
# Count how many routes each airport appears in
airport_route_counts = (
    pd.concat([
        route_df["ORIGIN"],
        route_df["DEST"]
    ])
    .value_counts()
)

# Keep only airports with at least this many routes - CAN CHANGE THIS VALUE IF WE WANT
min_routes = 10

major_airports = (
    airport_route_counts[airport_route_counts >= min_routes]
    .index
    .sort_values()
    .tolist()
)

airport_dropdown = widgets.Dropdown(
    options=["All Airports"] + major_airports,
    value="All Airports",
    description="Airport:"
)

def make_route_map(selected_airport):
    if selected_airport == "All Airports":
        filtered_routes = route_df.copy()
    else:
        filtered_routes = route_df[
            (route_df["ORIGIN"] == selected_airport) |
            (route_df["DEST"] == selected_airport)
        ].copy()

## map is essentially the same as above
    m = folium.Map(
        location=[39.5, -98.35],
        zoom_start=4,
        tiles="CartoDB positron",
        control_scale=True
    )

    colormap = cm.LinearColormap(
        colors=["#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"],
        vmin=route_df["avg_fare"].quantile(0.05),
        vmax=route_df["avg_fare"].quantile(0.95),
        caption="Average Fare ($)"
    )

    p90 = route_df["passengers"].quantile(0.90)

    for _, row in filtered_routes.iterrows():
        tooltip_text = (
            f"{row['ORIGIN']} → {row['DEST']} | "
            f"${row['avg_fare']:,.0f} | "
            f"{row['avg_distance']:,.0f} miles | "
            f"{row['passengers']:,.0f} passengers"
        )

        popup_html = f"""
        <div style="font-family: sans-serif; width: 220px;">
            <h4>{row['ORIGIN']} → {row['DEST']}</h4>
            <b>Fare:</b> ${row['avg_fare']:,.0f}<br>
            <b>Distance:</b> {row['avg_distance']:,.0f} mi<br>
            <b>Passengers:</b> {row['passengers']:,.0f}
        </div>
        """

        folium.PolyLine(
            locations=[
                [row["ORIGIN_LAT"], row["ORIGIN_LON"]],
                [row["DEST_LAT"], row["DEST_LON"]]
            ],
            color=colormap(row["avg_fare"]),
            weight=max(1.2, min(7, row["passengers"] / p90 * 5)),
            opacity=0.6,
            tooltip=tooltip_text,
            popup=folium.Popup(popup_html, max_width=260)
        ).add_to(m)

    airport_df = pd.concat([
        filtered_routes[["ORIGIN", "ORIGIN_LAT", "ORIGIN_LON"]]
            .rename(columns={"ORIGIN": "airport", "ORIGIN_LAT": "lat", "ORIGIN_LON": "lon"}),
        filtered_routes[["DEST", "DEST_LAT", "DEST_LON"]]
            .rename(columns={"DEST": "airport", "DEST_LAT": "lat", "DEST_LON": "lon"})
    ]).drop_duplicates()

    for _, row in airport_df.iterrows():
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=4,
            color="#1d3557",
            fill=True,
            fill_color="#1d3557",
            fill_opacity=0.9,
            weight=1,
            tooltip=row["airport"]
        ).add_to(m)

    colormap.add_to(m)
    return m


output = widgets.Output()

def update_map(change=None):
    with output:
        clear_output(wait=True)
        display(make_route_map(airport_dropdown.value))

airport_dropdown.observe(update_map, names="value")

display(airport_dropdown, output)
update_map()


Dropdown(description='Airport:', options=('All Airports', 'ATL', 'AUS', 'BNA', 'BOS', 'BWI', 'CLT', 'DCA', 'DE…

Output()